# Transformer Encoder-Decoder — IRB2400 Kinematics

## 무엇을 하는 노트북인가
IRB2400 6자유도 산업용 로봇팔에서
- **입력**: 과거 일정 시간의 **관절각 명령** `q1~q6` (6차원 시계열)
- **출력**: **다음 시점의 엔드이펙터 포즈** `(x, y, z, yaw, pitch, roll)` (6차원)
을 예측하는 **시퀀스-투-벡터(seq2one)** 회귀 모델이야.

## 왜 Transformer Encoder-Decoder 를 쓰나
기존에는 두 개의 노트북이 따로 있었어.
1. `tr_encoder_kinematics_pe.ipynb` — Encoder-only: 관절각 시퀀스만 bidirectional self-attention 으로 요약해서 다음 포즈를 예측
2. `tr_decoder_kinematics_pe.ipynb` — Decoder-only: 관절각 시퀀스를 causal self-attention 으로 요약 (미래 스텝을 보지 못하도록 마스킹)

이 노트북은 **둘을 합친 진짜 seq2seq 구조** 야. 핵심 아이디어는:
- Encoder 가 **명령 측 정보 (관절각)** 를 non-causal self-attention 으로 요약
- Decoder 가 **결과 측 정보 (과거 포즈)** 를 causal self-attention 으로 요약하면서,
  각 스텝마다 encoder 출력에 **cross-attention** 을 걸어 "어떤 관절 명령이 이 스텝의 포즈에 영향을 줬지?" 를 학습
- 마지막에 Dense head 로 **다음 시점 포즈** 를 예측

이렇게 하면 "과거 명령 + 과거 도달 포즈 → 다음 포즈" 라는 물리적으로도 말이 되는 모델링이 된다.
(실시간 제어 상황에서 시점 t 에는 과거 명령과 과거 측정 포즈가 모두 사용 가능하니까.)

## 타깃은 왜 단일 포즈인가
원래 두 노트북이 모두 `y[i+T]` 단일 벡터 6-dim 을 예측했어.
사용자가 "기존이랑 똑같이 예측해달라" 고 명시했기 때문에 출력 shape 은 `(batch, 6)` 으로 동일.


---
## [1] 라이브러리 임포트

- `pandas` / `numpy` : CSV 로딩과 배열 조작
- `matplotlib` : loss 곡선 / 예측 vs 실제 시각화
- `sklearn.model_selection.train_test_split` : 학습/검증 분할
- `sklearn.preprocessing.StandardScaler` : 입력·출력 정규화
  (MSE 기반 회귀에서는 스케일이 맞지 않으면 특정 차원이 손실을 지배해서 학습이 불안정해진다.
  예: `x` 는 수백 mm 단위, `yaw` 는 radian 단위 → 그대로 두면 위치 오차만 최적화됨)
- `sklearn.metrics` : MAE / RMSE / R² — 실제 단위 기준 해석이 쉬운 지표들
- `tensorflow.keras` : 함수형 API 로 Encoder-Decoder 를 구성
  - `MultiHeadAttention` : Transformer 의 핵심 연산
  - `LayerNormalization` : Transformer 블록 안정화 (BatchNorm 대신 사용하는 이유는 시퀀스 길이나 배치에 덜 민감하기 때문)
  - `GlobalAveragePooling1D` : `(batch, T, d) → (batch, d)` 로 시간축 평균 풀링
  - `EarlyStopping` : val_loss 정체 시 조기 종료 + 최고 가중치 복구


In [1]:
# [1] 라이브러리 임포트
import pandas as pd                                    # CSV 로딩용
import numpy as np                                     # 시퀀스 슬라이싱과 배열 연산
import matplotlib.pyplot as plt                        # 학습 곡선/예측 시각화

from sklearn.model_selection import train_test_split   # 학습/검증 분할
from sklearn.preprocessing import StandardScaler       # 정규화 (z-score). MSE 손실에서 스케일 맞추는 용도
from sklearn.metrics import (
    mean_absolute_error,                               # MAE: 해석이 직관적 ('평균적으로 얼마나 틀렸나')
    mean_squared_error,                                # RMSE 계산용 (큰 오차에 민감)
    r2_score,                                          # R²: 분산 설명도 (1에 가까울수록 좋음)
)

import tensorflow as tf                                # 텐서 연산과 커스텀 Layer 정의용
from tensorflow.keras.models import Model              # Functional API (다중 입력 모델이라 Sequential 로는 부족)
from tensorflow.keras.layers import (
    Input,                                             # 텐서 진입점
    Dense,                                             # FFN 과 출력 head
    Dropout,                                           # 정규화 기법 — 과적합 완화
    Add,                                               # residual connection (잔차 연결)
    LayerNormalization,                                # Transformer 의 표준 정규화
    GlobalAveragePooling1D,                            # 시간축 평균 풀링 → 단일 벡터로 축약
    MultiHeadAttention,                                # Transformer 의 핵심
)
from tensorflow.keras.callbacks import EarlyStopping   # 검증 손실 정체 시 조기 종료


ModuleNotFoundError: No module named 'pandas'

---
## [2] Google Drive 마운트

이 노트북은 Colab 전용 경로에 있는 CSV 를 읽을 거라 Drive 를 마운트해야 해.
로컬 Jupyter 로 돌릴 거면 이 셀은 건너뛰고 `csv_path` 만 로컬 경로로 바꿔주면 돼.


In [ ]:
# [2] Google Drive 마운트 (Colab 전용)
#    - 로컬에서 실행한다면 이 셀을 건너뛰고 [3] 의 csv_path 를 로컬 경로로 수정하면 됨.
from google.colab import drive
drive.mount('/content/drive')


---
## [3] 데이터 로딩

`datasetIRB2400.csv` 에는 시뮬레이션 또는 실제 로봇팔에서 수집한
`q1_in ~ q6_in`(입력 관절각)과 `x, y, z, yaw, pitch, roll`(실제 도달 포즈) 가 **시간 순서대로** 들어있어야 해.
행 순서가 곧 시간 축이라고 가정한다.

`print(df.head())` 로 컬럼 이름/값 범위를 눈으로 확인하는 이유:
- 정규화 전 각 차원의 스케일 차이를 체감하려고 (이게 StandardScaler 를 쓰는 이유)
- 혹시 NaN 이나 dtype 이 잘못되어 있으면 여기서 바로 발견하려고


In [ ]:
# [3] 데이터 로딩
csv_path = '/content/drive/MyDrive/Colab Notebooks/data/ngv/datasetIRB2400.csv'
df = pd.read_csv(csv_path)

print("데이터셋 로딩 완료:", df.shape)   # (N, 12) 형태일 것 — 관절각 6 + 포즈 6
print(df.head())                         # 컬럼 이름과 스케일 확인용


---
## [4]–[5] 입/출력 추출과 정규화

### 왜 StandardScaler (z-score) 인가
- **MinMaxScaler** 는 outlier 에 취약하고, attention softmax 이후의 통계가 뒤틀릴 수 있음
- **StandardScaler** 는 평균 0, 분산 1 로 맞춰 줘서
  - attention 의 dot-product 분포가 안정
  - MSE loss 가 모든 차원에 비슷한 기여를 하도록 만들어 줌
  - Adam optimizer 의 기본 learning rate 가 잘 작동하는 범위로 들어옴

### 왜 X 와 Y 를 따로 scaler 로 만드는가
- 입력과 타깃은 의미 자체가 다르기 때문에 **같은 scaler 로 합쳐서 fit 하면 안 됨**
- 예측값을 실제 단위로 되돌릴 때도 `scaler_Y.inverse_transform` 하나만 쓰면 돼서 깔끔함


In [ ]:
# [4] 입력/출력 컬럼 정의
X = df[['q1_in', 'q2_in', 'q3_in', 'q4_in', 'q5_in', 'q6_in']].values   # (N, 6) 관절각
Y = df[['x', 'y', 'z', 'yaw', 'pitch', 'roll']].values                   # (N, 6) 엔드이펙터 포즈

# [5] 정규화
#     - X 와 Y 는 물리 단위도 다르고 의미도 다르므로 각각 별도 scaler 로 fit 한다.
#     - fit_transform 은 train 전체에 대해 적용해도 이 노트북의 맥락에서는 OK:
#       시퀀스 분할 이후 어차피 train/test 로 나눌 것이고, 통계는 전체 데이터로 계산한 값이
#       시뮬레이션 기반 학습에는 실용적으로 충분하기 때문.
#       (엄밀한 누수 방지를 원한다면 train_test_split 이후 train 에만 fit 하면 됨.)
scaler_X = StandardScaler()
scaler_Y = StandardScaler()
X_scaled = scaler_X.fit_transform(X)
Y_scaled = scaler_Y.fit_transform(Y)


---
## [6] 시퀀스 생성 — 왜 두 개의 입력이 필요한가

이 노트북의 핵심 아이디어가 여기에 있어.

기존 encoder-only / decoder-only 노트북은 하나의 입력만 썼지만,
**Encoder-Decoder seq2seq** 는 두 종류의 시퀀스를 받는다:

| 이름            | 무엇       | shape       | 왜                                                                 |
|-----------------|-----------|-------------|--------------------------------------------------------------------|
| `encoder_input` | 과거 관절각 | (T, 6)     | 로봇에 보낸 **명령** 자체. 시간 순서 전체를 bidirectional 로 봐도 됨 |
| `decoder_input` | 과거 포즈  | (T, 6)     | 이미 **도달한 포즈 이력**. causal self-attn 으로 "내 이전까지의 궤적" 을 요약 |
| `target`        | 다음 포즈  | (6,)       | 예측하고 싶은 **다음 시점 포즈**                                      |

### 왜 decoder 입력으로 "과거 포즈" 를 쓰나
- 실시간 제어/예측 시나리오에서 **시점 t 에서는 과거 명령과 과거 측정 포즈가 모두 사용 가능**
- Decoder 의 cross-attention 은 "이 시점의 포즈 맥락에서 어떤 관절 명령이 중요했나?" 를 학습
- 즉 encoder 가 "source", decoder 가 "target-side context" 역할을 나눠 수행 (정통 seq2seq)

### 왜 sliding window 로 샘플을 만드나
- 시계열을 **지도학습 문제**로 바꾸려면 각 시점마다 (과거 T스텝, 다음 1스텝) 쌍을 만들어야 한다
- `time_steps=20`: 너무 짧으면 시간적 의존성을 못 잡고, 너무 길면 메모리·과적합 위험이 커짐. 20 은 경험적으로 무난한 값.


In [ ]:
# [6] 시퀀스 생성
#   - 각 샘플에 대해:
#       encoder_input = X[i : i+T]       관절각 과거 T스텝 (명령 측)
#       decoder_input = Y[i : i+T]       포즈 과거 T스텝 (결과 측, 측정 가능한 이력)
#       target        = Y[i + T]         바로 다음 시점의 포즈 (예측 대상)
#   - 시간축을 그대로 보존하기 위해 셔플 없이 순차적으로 만들어야 한다.
def create_sequences(X, Y, time_steps=20):
    enc_in, dec_in, tgt = [], [], []
    for i in range(len(X) - time_steps):
        enc_in.append(X[i : i + time_steps])         # (T, 6)
        dec_in.append(Y[i : i + time_steps])         # (T, 6)
        tgt.append(Y[i + time_steps])                # (6,)
    # np.array 로 묶으면 (N, T, 6), (N, T, 6), (N, 6) 이 된다.
    return np.array(enc_in), np.array(dec_in), np.array(tgt)

time_steps = 20    # 윈도우 길이. 너무 작으면 시간 의존성을 못 잡고, 너무 크면 과적합/계산량↑
enc_in, dec_in, y_seq = create_sequences(X_scaled, Y_scaled, time_steps)
print("encoder_input:", enc_in.shape)   # 기대: (N-T, 20, 6)
print("decoder_input:", dec_in.shape)   # 기대: (N-T, 20, 6)
print("target       :", y_seq.shape)    # 기대: (N-T, 6)


---
## [7] 학습/검증 분리

### 왜 세 배열을 **함께** split 하는가
`enc_in`, `dec_in`, `y_seq` 는 **같은 인덱스가 같은 샘플**을 의미한다.
따로따로 split 하면 "i번째 encoder 입력" 과 "i번째 target" 의 대응이 깨지니까,
반드시 한 번의 `train_test_split` 에 전부 넘겨서 **일치된 순열**로 나눠야 한다.

### 왜 `random_state=42`
재현성. 같은 코드를 돌렸을 때 다른 사람(또는 내일의 나)이 동일한 결과를 얻게 하려는 관례.

### 왜 `test_size=0.2`
데이터 20% 를 validation 으로 써서 학습 안정성을 보고 EarlyStopping 기준을 삼는다.
시계열에서 엄밀히 하려면 **시간순 holdout** (뒤쪽 20%) 을 쓰는 게 맞지만,
여기서는 기존 두 노트북과의 비교를 위해 동일하게 random split 을 사용한다.


In [ ]:
# [7] 학습/검증 분리 — 세 배열의 인덱스 정합을 지키려면 반드시 동시에 split 해야 한다.
enc_train, enc_test, dec_train, dec_test, y_train, y_test = train_test_split(
    enc_in, dec_in, y_seq,
    test_size=0.2,         # 80/20 split
    random_state=42,       # 재현성용 고정 시드
)
print("enc_train:", enc_train.shape, "dec_train:", dec_train.shape, "y_train:", y_train.shape)
print("enc_test :", enc_test.shape,  "dec_test :", dec_test.shape,  "y_test :", y_test.shape)


---
## [8] Positional Encoding — 왜 필요하고 왜 sin/cos 인가

### 왜 필요한가
Transformer 의 self-attention 은 **permutation-invariant** (순서를 바꿔도 결과가 같다).
시계열 예측에서 "t-1 에 발생한 일" 과 "t-10 에 발생한 일" 을 구분하려면
각 토큰에 **위치 정보**를 심어줘야 한다. 그게 Positional Encoding.

### 왜 sin/cos (사인파) 인가 — 학습 파라미터를 쓰지 않는 고전 방식
- **학습 파라미터가 0개** → 과적합 위험 없음, 시퀀스 길이를 바꿔도 작동
- 서로 다른 주파수의 sin/cos 을 쓰면 **임의의 두 위치 차이**를 attention 이 선형 변환으로 복원 가능 (원래 논문 "Attention Is All You Need" 의 의도)
- 학습형 PE (`Embedding`) 에 비해 데이터가 적을 때 일반화가 잘 됨

### 수식 요약
각 위치 `pos` 와 차원 인덱스 `i` 에 대해
- `PE(pos, 2k)   = sin(pos / 10000^(2k/d))`
- `PE(pos, 2k+1) = cos(pos / 10000^(2k/d))`

이 구현은 `angle_rates = 1 / 10000^(2*(i//2)/d)` 로 짝/홀 차원을 한번에 계산한 뒤
짝 인덱스에는 sin, 홀 인덱스에는 cos 을 넣어서 마지막에 concat 한다.


In [ ]:
# [8] Positional Encoding (encoder/decoder 양쪽에서 공용)
#     - 학습 파라미터 없음 (고정된 사인/코사인 함수)
#     - 입력 텐서에 그대로 덧셈 (broadcast) → 각 타임스텝에 위치 정보가 주입됨
class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, inputs):
        # inputs shape: (batch, T, d_model)
        seq_len = tf.shape(inputs)[1]           # 동적 shape 로 T 를 가져와야 eager/graph 둘 다 안전
        d_model = inputs.shape[-1]              # 정적 shape 로 충분 (모델 설계 시 고정됨)

        # 위치 인덱스 (T, 1) 과 차원 인덱스 (1, d_model)
        pos = tf.cast(tf.range(seq_len)[:, tf.newaxis], tf.float32)
        i   = tf.cast(tf.range(d_model)[tf.newaxis, :], tf.float32)

        # 원 논문의 스케일링: 주파수가 차원을 따라 지수적으로 감쇠
        # → 낮은 차원은 빠르게 변하는 신호, 높은 차원은 천천히 변하는 신호를 담게 됨
        angle_rates = 1 / tf.pow(10000.0, (2 * (i // 2)) / tf.cast(d_model, tf.float32))
        angle_rads  = pos * angle_rates                                  # (T, d_model)

        # 짝수 인덱스에는 sin, 홀수 인덱스에는 cos
        sines   = tf.sin(angle_rads[:, 0::2])
        cosines = tf.cos(angle_rads[:, 1::2])
        pos_encoding = tf.concat([sines, cosines], axis=-1)              # (T, d_model)
        pos_encoding = pos_encoding[tf.newaxis, ...]                     # (1, T, d_model) — 배치 축 broadcast
        return inputs + pos_encoding                                     # 위치 정보 주입


---
## [9] Encoder Block — Pre-Norm Transformer

### 구조
```
x ─┬──────── LayerNorm ── MultiHeadAttention(self) ──┬── +x ─┬──────── LayerNorm ── Dense(ff) ── Dropout ── Dense(d) ──┬── +x ── out
   │                                                │       │                                                       │
   └─────────────── residual ──────────────────────┘       └────────────────────── residual ─────────────────────┘
```

### 왜 **Pre-Norm** (LN → Attn) 인가
- 원 Transformer 는 Post-Norm (Attn → Add → LN) 이지만,
  깊게 쌓을수록 gradient 가 불안정해지는 문제가 있음
- **Pre-Norm** 은 residual path 가 항상 "raw 입력" 을 유지하므로 학습이 안정적
- 최근 대부분의 구현체 (GPT-2 이후, Llama 등) 가 Pre-Norm 을 사용

### 왜 residual connection 인가
- 깊은 네트워크에서 gradient vanishing 방지
- Attention/FFN 이 "입력에 대한 **변화량**" 만 학습하면 되므로 최적화가 쉬워짐

### 왜 FFN 중간에 Dropout 만 넣고 앞뒤에는 안 넣는가
- Attention 출력은 이미 MultiHeadAttention 내부의 `dropout` 옵션으로 정규화됨
- FFN 의 hidden 에서만 추가 dropout → 과적합 방지하면서 표현력은 유지


In [ ]:
# [9] Encoder block — bidirectional self-attention + FFN (Pre-Norm)
#     - 관절각 시퀀스는 미래/과거를 자유롭게 참조해도 됨 (non-causal)
#     - 따라서 attention_mask 를 걸지 않는다.
def encoder_block(x, head_size=32, num_heads=4, ff_dim=128, dropout=0.1):
    # ---- 1) Self-Attention sub-layer (Pre-Norm) ----
    a = LayerNormalization(epsilon=1e-6)(x)                              # 먼저 정규화
    a = MultiHeadAttention(                                              # Q=K=V=x → self-attention
        key_dim=head_size,                                               # head 당 key/query 차원
        num_heads=num_heads,                                             # 여러 '관점'으로 attend
        dropout=dropout,                                                 # attention weight 에 dropout
    )(a, a)                                                              # 마스크 없음 = bidirectional
    x = Add()([x, a])                                                    # residual: 입력을 보존

    # ---- 2) Feed-Forward sub-layer (Pre-Norm) ----
    f = LayerNormalization(epsilon=1e-6)(x)
    f = Dense(ff_dim, activation='relu')(f)                              # 확장 (보통 d_model 의 2~4배)
    f = Dropout(dropout)(f)                                              # FFN hidden 에 dropout
    f = Dense(x.shape[-1])(f)                                            # 다시 d_model 차원으로 투영
    return Add()([x, f])                                                 # residual


---
## [10] Decoder Block — Causal Self-Attn + Cross-Attn + FFN

Decoder 는 encoder 보다 **한 단계 더** 있어. 세 개의 sub-layer 로 구성:

1. **Masked (causal) Self-Attention**
   - 현재 스텝은 **과거 스텝만** 참조 (미래 누수 방지)
   - 학습 시 타깃 측 정보가 앞쪽으로 흐르지 않도록 하는 정통 seq2seq 제약
2. **Cross-Attention (Encoder-Decoder Attention)**
   - Query 는 decoder 쪽에서 (현재 포즈 맥락), Key/Value 는 encoder 쪽에서 (관절각 시퀀스 요약)
   - "이 시점의 포즈를 예측하려면 어떤 관절 명령을 강조해야 하나?" 를 학습
3. **Feed-Forward**
   - encoder 와 동일한 의미: point-wise 변환으로 표현력 추가

### 왜 여기서는 **Post-Norm** 인가
- 기존 `tr_decoder_kinematics_pe.ipynb` 의 스타일을 그대로 따름
- Encoder(Pre-Norm) 와 Decoder(Post-Norm) 가 혼합되어 있지만, 얕은 (2블록) 네트워크에서는 실험적으로 둘 다 잘 작동
- 만약 더 깊게 쌓을 거면 Decoder 도 Pre-Norm 으로 통일하는 게 권장됨

### Causal Mask 구현 메모
`tf.linalg.band_part(ones, -1, 0)` = **하삼각 행렬**
```
1 0 0 0
1 1 0 0
1 1 1 0
1 1 1 1
```
이걸 attention score 에 곱해서 "각 row(query) 는 자기 이하의 col(key) 만 참조" 하도록 제한.


In [ ]:
# [10] Decoder block — Masked Self-Attention + Cross-Attention + FFN

# Causal (미래 차단) self-attention 을 MultiHeadAttention 래퍼로 감싼 Layer
class CausalSelfAttention(tf.keras.layers.Layer):
    def __init__(self, head_size, num_heads, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.attn = MultiHeadAttention(
            key_dim=head_size,
            num_heads=num_heads,
            dropout=dropout,
        )

    def call(self, x):
        seq_len = tf.shape(x)[1]
        # 하삼각 mask: (T, T) — 1 은 '볼 수 있음', 0 은 '마스킹'
        causal_mask = tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
        # MultiHeadAttention 이 요구하는 shape (1, 1, T, T) 로 brodcast 대비
        causal_mask = causal_mask[tf.newaxis, tf.newaxis, :, :]
        return self.attn(x, x, attention_mask=causal_mask)


def decoder_block(x, enc_out, head_size=32, num_heads=4, ff_dim=128, dropout=0.1):
    # ---- 1) Masked (causal) Self-Attention ----
    #     - decoder 자신의 과거 포즈 시퀀스를 요약 (미래 마스킹)
    a = CausalSelfAttention(head_size, num_heads, dropout)(x)
    x = Add()([x, a])
    x = LayerNormalization(epsilon=1e-6)(x)

    # ---- 2) Cross-Attention ----
    #     - query : decoder 의 현재 상태 (현재 포즈 맥락)
    #     - key/value : encoder 출력 (관절각 시퀀스의 요약)
    #     - 이 블록이 바로 encoder 와 decoder 를 '연결' 하는 핵심이다.
    c = MultiHeadAttention(
        key_dim=head_size, num_heads=num_heads, dropout=dropout,
    )(query=x, key=enc_out, value=enc_out)                              # mask 없음 — encoder 는 전부 봐도 됨
    x = Add()([x, c])
    x = LayerNormalization(epsilon=1e-6)(x)

    # ---- 3) Feed-Forward ----
    f = Dense(ff_dim, activation='relu')(x)
    f = Dropout(dropout)(f)
    f = Dense(x.shape[-1])(f)
    x = Add()([x, f])
    return LayerNormalization(epsilon=1e-6)(x)


---
## [11] 모델 구성 — 전체 파이프라인 조립

```
        encoder_input (T,6)                          decoder_input (T,6)
             │                                           │
             ▼                                           ▼
      PositionalEncoding                          PositionalEncoding
             │                                           │
        encoder_block × 2                         decoder_block × 2
             │                                      (cross-attn 으로
             │                                       encoder 출력 참조)
             ▼                                           │
       encoder_output  ───────── cross-attn ────────────┘
                                                         │
                                              GlobalAveragePooling1D
                                                         │
                                                   Dense(64, relu)
                                                         │
                                                    Dropout(0.2)
                                                         │
                                                     Dense(6)
                                                         │
                                                    next_pose (6,)
```

### 하이퍼파라미터 의미
- `head_size=32`, `num_heads=4` → attention 차원은 총 `32 × 4 = 128`. 작은 데이터셋에 적당히 큰 표현력.
- `ff_dim=128` → FFN hidden 차원. 보통 d_model 의 2~4배지만 여기선 6(d_model) 에 비하면 훨씬 큼. 6 은 너무 작아서 FFN 은 여유롭게 잡음.
- `dropout=0.1` → 과적합 완화
- `GlobalAveragePooling1D` → (batch, T, d) 를 (batch, d) 로 시간축 평균. **마지막 토큰만 쓰는 것** 보다 안정적이고 회귀 문제에 적합.
- 회귀 head: `Dense(64, relu) → Dropout(0.2) → Dense(6)` — 기존 decoder 노트북과 동일한 head 사이즈.
- **출력 activation 없음** → 회귀이므로 선형(identity) 출력을 쓰고 MSE 로 학습.

### 왜 Adam + MSE
- **Adam** : 하이퍼파라미터 기본값만으로도 잘 동작하는 표준 옵티마이저
- **MSE (Mean Squared Error)** : 회귀의 정석. 큰 오차에 더 민감해서 모델이 outlier 를 줄이는 방향으로 학습
- **MAE 는 metric 으로만** : 학습 중 진행 상황을 "평균 오차 단위" 로 직관적으로 확인


In [ ]:
# [11] 모델 구성 (Encoder + Decoder + Pooling + Dense head)

# ---- 하이퍼파라미터 ----
head_size = 32     # head 당 key/query 차원
num_heads = 4      # head 수 → 총 attention dim = head_size * num_heads = 128
ff_dim    = 128    # FFN hidden 차원
dropout   = 0.1    # attention / FFN dropout 비율

enc_input_shape = (time_steps, enc_train.shape[2])   # (20, 6) — 관절각 시퀀스
dec_input_shape = (time_steps, dec_train.shape[2])   # (20, 6) — 과거 포즈 시퀀스

# ---- 두 개의 Input (다중 입력이므로 Functional API 필수) ----
enc_inputs = Input(shape=enc_input_shape, name='encoder_input')
dec_inputs = Input(shape=dec_input_shape, name='decoder_input')

# ---- Encoder stack : 관절각 시퀀스를 non-causal self-attention 으로 요약 ----
e = PositionalEncoding(name='enc_pe')(enc_inputs)
e = encoder_block(e, head_size, num_heads, ff_dim, dropout)
e = encoder_block(e, head_size, num_heads, ff_dim, dropout)

# ---- Decoder stack : 과거 포즈 시퀀스에 대해 causal self-attn + encoder 출력에 cross-attn ----
d = PositionalEncoding(name='dec_pe')(dec_inputs)
d = decoder_block(d, e, head_size, num_heads, ff_dim, dropout)
d = decoder_block(d, e, head_size, num_heads, ff_dim, dropout)

# ---- 회귀 head ----
#   - GlobalAveragePooling1D : 마지막 토큰만 쓰는 것보다 평균이 회귀에 더 안정적
#   - Dense(64,relu) + Dropout : 소규모 비선형 head
#   - Dense(6) : activation 없음 — 포즈는 실수 벡터이므로 선형 출력
h = GlobalAveragePooling1D()(d)
h = Dense(64, activation='relu')(h)
h = Dropout(0.2)(h)
outputs = Dense(6, name='next_pose')(h)

# ---- 모델 컴파일 ----
#   - optimizer='adam' : 기본값으로도 잘 수렴하는 표준 선택
#   - loss='mse' : 회귀 문제의 기본. 큰 오차를 더 크게 벌함
#   - metrics=['mae'] : 학습 중 '평균 절대 오차' 를 모니터링 (해석 쉬움)
model = Model(inputs=[enc_inputs, dec_inputs], outputs=outputs)
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()


---
## [12] 학습

### 왜 EarlyStopping
- Transformer 는 소규모 데이터에서 **쉽게 과적합** 한다
- `patience=5` : val_loss 가 5 epoch 동안 개선이 없으면 중단
- `restore_best_weights=True` : 멈춘 시점이 아닌 **가장 좋았던 가중치** 로 복구 → 실제로 쓰는 모델 품질↑

### 왜 `batch_size=64`, `epochs=20`
- 배치 64 : GPU 메모리 여유 있게 쓰면서도 gradient noise 를 적절히 유지
- 최대 20 epoch : EarlyStopping 이 보통 그 전에 멈추게 됨. 충분한 상한선.

### 두 입력을 리스트로 넘기는 방식
`model.fit([enc_train, dec_train], y_train, ...)` — 다중 Input 모델은 입력을 **리스트** 또는 **dict** 로 전달한다.
dict 로 넘기면 `{'encoder_input': ..., 'decoder_input': ...}` 처럼 이름으로 매칭할 수도 있음.


In [ ]:
# [12] 학습
# EarlyStopping:
#   - Transformer 는 소규모 데이터에서 쉽게 과적합하므로 검증 손실 기반 조기 종료는 거의 필수
#   - restore_best_weights=True 로 최종 가중치를 '최고 성능' 시점으로 되돌린다
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
)

history = model.fit(
    [enc_train, dec_train], y_train,                  # 다중 입력이므로 리스트로 전달
    validation_data=([enc_test, dec_test], y_test),   # validation 도 동일하게 리스트
    epochs=20,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1,
)


---
## [13] 예측 및 역정규화

모델은 정규화된 공간에서 학습했기 때문에 출력도 정규화된 값이야.
실제 단위 (mm, rad 등) 로 해석하려면 `scaler_Y.inverse_transform` 으로 되돌려야 한다.
`y_true` 도 똑같이 역변환해야 오차 계산이 **같은 좌표계** 에서 이루어진다.


In [ ]:
# [13] 예측 및 역정규화
y_pred_scaled = model.predict([enc_test, dec_test])      # 정규화된 공간의 예측
y_pred = scaler_Y.inverse_transform(y_pred_scaled)       # 실제 단위로 복원
y_true = scaler_Y.inverse_transform(y_test)              # 정답도 동일하게 복원 (둘 다 같은 공간에서 비교)


---
## [14] 평가 지표

세 가지 지표를 같이 보는 이유:

| 지표 | 의미 | 장점 |
|------|------|------|
| **MAE** | 평균 절대 오차 | 실제 단위 그대로 해석 ("평균 x mm 만큼 틀린다") |
| **RMSE** | 평균 제곱근 오차 | 큰 오차에 민감 — outlier 가 있는지 확인 용이 |
| **R²** | 분산 설명도 | 1에 가까울수록 좋음. 단위 없는 통일 지표 |

### 차원별 MAE 도 출력하는 이유
전체 MAE 는 6차원 평균이라 어떤 차원이 특히 취약한지 숨겨져 있음.
`x`, `y`, `z` (mm) 와 `yaw`, `pitch`, `roll` (rad) 는 단위 자체가 달라서
각각 따로 보는 게 모델 진단에 훨씬 유용하다.


In [ ]:
# [14] 평가지표 (MAE / RMSE / R²)
mae  = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))       # MSE 의 제곱근 → 원래 단위
r2   = r2_score(y_true, y_pred)                          # 1에 가까울수록 분산을 잘 설명

print(f"실제 단위 기준 MAE : {mae:.4f}")
print(f"실제 단위 기준 RMSE: {rmse:.4f}")
print(f"R^2               : {r2:.4f}")

# 각 차원별 MAE — 위치(mm)와 자세(rad)는 단위가 다르므로 분리해서 보는 게 유용
columns = ['x', 'y', 'z', 'yaw', 'pitch', 'roll']
mae_each = np.mean(np.abs(y_true - y_pred), axis=0)
for name, value in zip(columns, mae_each):
    print(f"  {name:<5s} MAE: {value:.4f}")


---
## [15] 학습 곡선 시각화

**train loss vs val loss** 를 같이 그리는 이유:
- 둘 다 감소 → 정상 학습
- train 만 감소, val 은 증가 → **과적합** 시작 (EarlyStopping 이 멈췄을 것)
- 둘 다 정체 → 학습률이나 모델 용량을 재검토


In [ ]:
# [15] 학습 곡선
plt.figure()
plt.plot(history.history['loss'],     label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss (MSE, scaled)')
plt.title('Encoder-Decoder Loss Curve')
plt.grid(True); plt.legend(); plt.show()


---
## [16] 예측 vs 실제 — x 좌표 예시

MAE/RMSE 숫자만으로는 **어떤 패턴에서 모델이 틀리는지** 보이지 않는다.
실제 값과 예측 값을 같이 그리면:
- 전체적으로 스케일이 맞는지
- 급격한 변화 구간에서 뒤처지는지 (lag)
- 특정 구간에서만 편향(bias) 이 있는지
한 눈에 확인 가능.

여기서는 x 좌표만 그렸지만, 다른 차원도 같은 방식으로 바꿔 보며 디버깅하면 된다.


In [ ]:
# [16] 예측 vs 실제 (x 좌표 예시)
#   - 다른 차원을 보고 싶으면 인덱스(0)를 1~5 로 바꾸면 된다:
#       0: x, 1: y, 2: z, 3: yaw, 4: pitch, 5: roll
plt.figure()
plt.plot(y_true[:, 0], label='True x')
plt.plot(y_pred[:, 0], label='Pred x')
plt.xlabel('Sample Index'); plt.ylabel('x')
plt.title('True vs Predicted — x')
plt.grid(True); plt.legend(); plt.show()
